In [8]:
import pandas as pd
import numpy as np
df = pd.read_csv(r"C:\Users\varad\Downloads\Automated-Transaction-ETL-Pipeline\data\raw\upi_transactions_2024.csv")
pd.set_option('display.max_columns', None)
print(df.head())

  transaction id            timestamp transaction type merchant_category  \
0  TXN0000000001  2024-10-08 15:17:28              P2P     Entertainment   
1  TXN0000000002  2024-04-11 06:56:00              P2M           Grocery   
2  TXN0000000003  2024-04-02 13:27:18              P2P           Grocery   
3  TXN0000000004  2024-01-07 10:09:17              P2P              Fuel   
4  TXN0000000005  2024-01-23 19:04:23              P2P          Shopping   

   amount (INR) transaction_status sender_age_group receiver_age_group  \
0           868            SUCCESS            26-35              18-25   
1          1011            SUCCESS            26-35              26-35   
2           477            SUCCESS            26-35              36-45   
3          2784            SUCCESS            26-35              26-35   
4           990            SUCCESS            26-35              18-25   

    sender_state sender_bank receiver_bank device_type network_type  \
0          Delhi        Axi

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 17 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   transaction id      250000 non-null  object
 1   timestamp           250000 non-null  object
 2   transaction type    250000 non-null  object
 3   merchant_category   250000 non-null  object
 4   amount (INR)        250000 non-null  int64 
 5   transaction_status  250000 non-null  object
 6   sender_age_group    250000 non-null  object
 7   receiver_age_group  250000 non-null  object
 8   sender_state        250000 non-null  object
 9   sender_bank         250000 non-null  object
 10  receiver_bank       250000 non-null  object
 11  device_type         250000 non-null  object
 12  network_type        250000 non-null  object
 13  fraud_flag          250000 non-null  int64 
 14  hour_of_day         250000 non-null  int64 
 15  day_of_week         250000 non-null  object
 16  is

In [18]:
df.duplicated().sum()
print(df.shape)
print(df.isnull().sum())

(250000, 17)
transaction id        0
timestamp             0
transaction type      0
merchant_category     0
amount (INR)          0
transaction_status    0
sender_age_group      0
receiver_age_group    0
sender_state          0
sender_bank           0
receiver_bank         0
device_type           0
network_type          0
fraud_flag            0
hour_of_day           0
day_of_week           0
is_weekend            0
dtype: int64


In [21]:
print(df['merchant_category'].value_counts())
print(df['transaction_status'].value_counts())
print(df["transaction type"].value_counts())
print(df["device_type"].value_counts())
print(df["network_type"].value_counts())
print(df["fraud_flag"].value_counts())


merchant_category
Grocery          49966
Food             37464
Shopping         29872
Fuel             25063
Other            24828
Utilities        22338
Transport        20105
Entertainment    20103
Healthcare       12663
Education         7598
Name: count, dtype: int64
transaction_status
SUCCESS    237624
FAILED      12376
Name: count, dtype: int64
transaction type
P2P             112445
P2M              87660
Bill Payment     37368
Recharge         12527
Name: count, dtype: int64
device_type
Android    187777
iOS         49613
Web         12610
Name: count, dtype: int64
network_type
4G      149813
5G       62582
WiFi     25134
3G       12471
Name: count, dtype: int64
fraud_flag
0    249520
1       480
Name: count, dtype: int64


In [16]:
df.describe()

,amount (INR),fraud_flag,hour_of_day,is_weekend
count,250000.000000,250000.000000,250000.000000,250000.000000
mean,1311.756036,0.001920,14.681032,0.285348
std,1848.059224,0.043776,5.188304,0.451581
min,10.000000,0.000000,0.000000,0.000000
25%,288.000000,0.000000,11.000000,0.000000
50%,629.000000,0.000000,15.000000,0.000000
75%,1596.000000,0.000000,19.000000,1.000000
max,42099.000000,1.000000,23.000000,1.000000


In [22]:
df["sender_bank"].value_counts()

sender_bank
SBI         62693
HDFC        37485
ICICI       29769
IndusInd    25173
Axis        25042
PNB         24946
Yes Bank    24860
Kotak       20032
Name: count, dtype: int64

In [23]:
df["receiver_bank"].value_counts()

receiver_bank
SBI         62378
HDFC        37651
ICICI       29944
IndusInd    25086
Yes Bank    25009
Axis        24992
PNB         24802
Kotak       20138
Name: count, dtype: int64

In [24]:
# Create a copy so original dataframe remains unchanged
df_transformed = df.copy()

In [25]:
df_transformed["timestamp"] = pd.to_datetime(df_transformed["timestamp"])
print(df_transformed["timestamp"].dtype)
df_transformed["transaction_date"] = df_transformed["timestamp"].dt.date
df_transformed["transaction_month"] = (
    df_transformed["timestamp"].dt.to_period("M").astype(str)
)
df_transformed["transaction_quarter"] = (
    "Q" + df_transformed["timestamp"].dt.quarter.astype(str)
)
bins = [0, 500, 1000, 5000, 10000, float("inf")]

labels = [
    "Micro Transaction",
    "Small Transaction",
    "Medium Transaction",
    "Large Transaction",
    "Premium Transaction"
]

df_transformed["amount_bucket"] = pd.cut(
    df_transformed["amount (INR)"],
    bins=bins,
    labels=labels
)
print(df_transformed["amount_bucket"].value_counts())
df_transformed["fraud_status"] = (
    df_transformed["fraud_flag"]
    .map({0: "Genuine", 1: "Fraud"})
)
print(df_transformed["fraud_status"].value_counts())
df_transformed["transaction_outcome"] = (
    df_transformed["transaction_status"]
    .map({
        "SUCCESS": "Successful",
        "FAILED": "Failed"
    })
)
print(df_transformed["transaction_outcome"].value_counts())


datetime64[ns]
amount_bucket
Micro Transaction      106624
Medium Transaction      81383
Small Transaction       51037
Large Transaction        9151
Premium Transaction      1805
Name: count, dtype: int64
fraud_status
Genuine    249520
Fraud         480
Name: count, dtype: int64
transaction_outcome
Successful    237624
Failed         12376
Name: count, dtype: int64


In [30]:
df_transformed.columns
# df_transformed.head()

Index(['transaction id', 'timestamp', 'transaction type', 'merchant_category',
       'amount (INR)', 'transaction_status', 'sender_age_group',
       'receiver_age_group', 'sender_state', 'sender_bank', 'receiver_bank',
       'device_type', 'network_type', 'fraud_flag', 'hour_of_day',
       'day_of_week', 'is_weekend', 'transaction_date', 'transaction_month',
       'transaction_quarter', 'amount_bucket', 'fraud_status',
       'transaction_outcome', 'week_type'],
      dtype='object')

In [29]:
df_transformed["week_type"] = (
    df_transformed["is_weekend"]
    .map({
        0: "Weekday",
        1: "Weekend"
    })
)